# VRI 2026 - YOLO Platform & Markers Tracker

This notebook unzips the `colab_dataset.zip` from your Google Drive, installs Ultralytics, and trains the YOLOv8-Pose model on the combined manual and synthetic datasets.

In [ ]:
# Mount Google Drive so we can access the dataset and save the .pt weights
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the dataset and install Ultralytics
!unzip -q -o /content/drive/MyDrive/colab_dataset.zip -d /content/
!pip install ultralytics
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Train the model natively in Python
from ultralytics import YOLO
import os

yaml_path = '/content/dataset/colab_dataset.yaml'
if not os.path.exists(yaml_path):
    print(f"ERROR: Could not find {yaml_path}. Make sure the dataset was unzipped correctly!")
else:
    model = YOLO('yolov8n-pose.pt')
    
    project_dir = '/content/drive/MyDrive/YOLO_Models'
    os.makedirs(project_dir, exist_ok=True)
    
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=16,
        project='models',
        name='yolov8_platform_pose_markers_v3',
        save_dir=project_dir,
        exist_ok=True,
        # Heavy augmentations
        perspective=0.001, # Perspective warp
        fliplr=0.5,
        degrees=90.0,      # Rotations (increased for rotational invariance)
        translate=0.2,     # Horizontal/vertical shift to improve off-center platform center predictions
        scale=0.9,         # Zoom out/in by 90% (increased for distance invariance)
        mosaic=1.0,        # High mosaic for background variety
        hsv_h=0.015,       # Color jitter (Hue)
        hsv_s=0.7,         # Color jitter (Sat)
        hsv_v=0.4          # Color jitter (Val)
    )
    print(f"Training complete! Model saved in {project_dir}/yolov8_platform_pose_markers_v3/weights/best.pt")
